# 📖 우르시누스 하이델베르크 교리문답 대작 주석 영한 대조 번역기
### *The Commentary of Dr. Zacharias Ursinus on the Heidelberg Catechism (1851)*

- **원저:** 자카리아스 우르시누스 (Zacharias Ursinus, 1534~1583)
- **영역:** G. W. Williard (1851년 미국 초판, 프린스턴 신학교 도서관 소장본)
- **저작권:** 1851년 출간으로 전 세계 **완전 공공도메인 (Public Domain)**, 저작권 만료
- **특징:** 구글 드라이브에 원서를 초고속 다이렉트로 다운받고, **문단별 영한 대조(한글 완역 우선 + 영어 원문 접기)**로 읽기 쉽게 번역해 줍니다.


In [ ]:
# [1단계] 필수 번역 라이브러리 설치 및 구글 드라이브 연결
!pip install -q deep-translator tqdm

from google.colab import drive
import os
import requests
from deep_translator import GoogleTranslator
from tqdm import tqdm

print("구글 드라이브를 연결합니다. 팝업이 뜨면 [Google Drive에 연결]을 눌러주세요...")
drive.mount("/content/drive")

# 구글 드라이브 저장 폴더 생성
save_dir = "/content/drive/MyDrive/신학연구/우르시누스_하이델베르크"
os.makedirs(save_dir, exist_ok=True)
print(f"
 저장 위치 준비 완료: {save_dir}")


In [ ]:
# [2단계] 저작권 만료 우르시누스 주석 텍스트(OCR) 다운로드
# Internet Archive 프린스턴 신학대학원 소장본 공식 주소
txt_url = "https://archive.org/download/commentaryofdrza00ursi/commentaryofdrza00ursi_djvu.txt"
pdf_url = "https://archive.org/download/commentaryofdrza00ursi/commentaryofdrza00ursi.pdf"

local_txt = os.path.join(save_dir, "ursinus_full_text.txt")

if not os.path.exists(local_txt):
    print("인터넷 아카이브에서 우르시누스 주석 텍스트를 다운로드합니다 (구글 서버 전송 약 5초 소요)...")
    !wget -q -O "{local_txt}" "{txt_url}"
    print(f" 텍스트 다운로드 완료: {local_txt}")
else:
    print(f" 이미 텍스트 파일이 준비되어 있습니다: {local_txt}")

print("
(참고) 1.2GB 분량의 원서 PDF 원본 스캔본도 드라이브에 함께 보관하시려면 아래 줄 앞의 #을 지우고 실행하세요:")
print(f'# !wget -O "{os.path.join(save_dir, "ursinus_original_scan.pdf")}" "{pdf_url}"')


In [ ]:
# [3단계] 영한 대조 번역 실행 (원하는 분량 지정)
def translate_bilingual(input_file, output_file, max_paragraphs=30, start_paragraph=0):
    translator = GoogleTranslator(source="en", target="ko")
    
    with open(input_file, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    
    # 의미 있는 문단 분리 (빈 줄 기준, 너무 짧은 헤더 제외)
    raw_paragraphs = [p.strip() for p in content.split("

") if len(p.strip()) > 50]
    
    selected_paragraphs = raw_paragraphs[start_paragraph : start_paragraph + max_paragraphs]
    total = len(selected_paragraphs)
    
    print(f"총 {total}개 문단을 영한 대조로 번역합니다 (문단 {start_paragraph + 1}번 ~ {start_paragraph + total}번)...
")
    
    bilingual_lines = [
        "# 📖 우르시누스 하이델베르크 교리문답 대작 주석 (영한 대조)
",
        "> 원저: Zacharias Ursinus (1534-1583) | 영역: G. W. Williard (1851)
",
        f"> 번역 범위: 문단 {start_paragraph + 1} ~ {start_paragraph + total}

",
        "---

"
    ]
    
    for idx, para in enumerate(tqdm(selected_paragraphs, desc="번역 진행 중")):
        clean_para = " ".join(para.split())
        
        # 글자 수가 길면 분할 번역 (번역기 오류 방지)
        chunks = []
        if len(clean_para) > 2500:
            sentences = clean_para.split(". ")
            curr = ""
            for s in sentences:
                if len(curr) + len(s) < 2500:
                    curr += s + ". "
                else:
                    chunks.append(curr.strip())
                    curr = s + ". "
            if curr:
                chunks.append(curr.strip())
        else:
            chunks = [clean_para]
            
        ko_parts = []
        for c in chunks:
            try:
                ko = translator.translate(c)
                ko_parts.append(ko)
            except Exception as e:
                ko_parts.append(f"[번역 오류: {e}]")
                
        full_ko = " ".join(ko_parts)
        
        # 마크다운 영한 대조 서식
        bilingual_lines.append(f"### [문단 {start_paragraph + idx + 1}]

")
        bilingual_lines.append(f"**🇰🇷 한글 번역:**

{full_ko}

")
        bilingual_lines.append(f"<details><summary><b>🇺🇸 영어 원문 보기 (클릭하면 펼쳐짐)</b></summary>

> {clean_para}

</details>

")
        bilingual_lines.append("---

")
        
    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(bilingual_lines)
        
    print(f"
 번역 완료! 결과물이 구글 드라이브에 마크다운(.md) 파일로 저장되었습니다:")
    print(f" {output_file}")

# [실행] 처음 30개 문단(서론 및 제1문답) 번역
output_md = os.path.join(save_dir, "우르시누스_주석_영한대조_01.md")
translate_bilingual(local_txt, output_md, max_paragraphs=30, start_paragraph=0)


## 💡 목사님을 위한 활용 팁
1. 번역된 파일은 구글 드라이브 **** 폴더에 자동 저장됩니다.
2. 구글 문서(Google Docs)나 메모장, 또는 목사님의 디지털 서재 리더에서 바로 열어보실 수 있습니다.
3. 다음 분량을 번역하시려면 맨 아래 코드에서 처럼 숫자를 변경하고 다시 실행하시면 됩니다!
